In [2]:
import pandas as pd
import multiprocessing as mp
from ete3 import Tree
import pickle
from ete3 import NCBITaxa, Tree

## Make a table to assist with tree manual inspection

In [ ]:

def init_worker():
    global ncbi
    ncbi = NCBITaxa()
ncbi = NCBITaxa()


def get_ncbi_lineage_by_rank(species_name, prefer_rank="species"):

    """
    Return NCBI taxid and full lineage as {rank: scientific_name}
    for a species name.
    """
    global ncbi
    try:
        if ncbi is None:
            # Fallback for non-multiprocessing use
            ncbi = NCBITaxa()
        species_name = species_name.strip()

        if not species_name:
            raise ValueError("species_name is empty")
        name2taxids = ncbi.get_name_translator([species_name])


        if species_name not in name2taxids:
            if len(species_name.split(" "))>2:
                species_name=" ".join(species_name.split(" ")[0:2])
                get_ncbi_lineage_by_rank(" ".join(species_name.split(" ")[0:2]))
                try:
                    return get_ncbi_lineage_by_rank(" ".join(species_name.split(" ")[0:2]))
                except:
                    print(species_name)
                    return 'None','None'
            raise ValueError(f"No NCBI taxid found for: {species_name!r}")

        candidate_taxids = name2taxids[species_name]

        if len(candidate_taxids) > 1:
            candidate_ranks = ncbi.get_rank(candidate_taxids)
            preferred = [
                taxid for taxid in candidate_taxids
                if candidate_ranks.get(taxid) == prefer_rank
            ]
            taxid = preferred[0] if preferred else candidate_taxids[0]
        else:
            taxid = candidate_taxids[0]

        lineage_taxids = ncbi.get_lineage(taxid)
        lineage_names = ncbi.get_taxid_translator(lineage_taxids)
        lineage_ranks = ncbi.get_rank(lineage_taxids)

        lineage_by_rank = {
            lineage_ranks[t]: lineage_names[t]
            for t in lineage_taxids
            if lineage_ranks.get(t) not in (None, "no rank")
        }
        return taxid, lineage_by_rank
    except:
        return 'None',{'None':'None'}


In [ ]:

alldf=pd.read_csv("outputs/per_protein_diamond_data.tsv",sep="\t",index_col=0)
dipt_proteins=set(alldf[alldf.slineage.astype(str).str.contains('Diptera')]['accession'])
htgdf=alldf[alldf.HGT]
htgdf=htgdf.set_index('accession')
htgs=set(htgdf.index)

import pickle
with open('outputs/connected_components.pickle', 'rb') as f:
    connected_components = pickle.load(f)
with open('outputs/connected_components_reps.pickle', 'rb') as f:
    representative = pickle.load(f)

In [ ]:
def get_sister_info(ranks):
    domain = kingdom = phylum = classt = order = None
    try:
        if 'Viruses' in str(ranks):
            domain='Viruses'
        else:
            domain=ranks['domain']
    except:
        superkingdom='None'
    try:
        kingdom=ranks['kingdom']
    except:
        kingdom='None'
    try:
        phylum=ranks['phylum']
    except:
        phylum='None'
    try:
        classt=ranks['class']
    except:
        classt='None'
    try:
        order=ranks['order']
    except:
        order='None'
    return [domain, kingdom, phylum, classt, order]


In [ ]:
def get_sister_cousin(node):

    parent = node.up
    grandparent = parent.up if parent is not None else None

    # Sister clades: children of parent that are not the focal node
    sisters = []
    if parent is not None:
        for child in parent.children:
            if child is not node:
                sisters.extend(child.iter_leaves())

    # Cousin clades: children of grandparent that are not the parent clade
    cousins = []
    if grandparent is not None:
        for child in grandparent.children:
            if child is not parent:
                cousins.extend(child.iter_leaves())
    return sisters, cousins

In [ ]:
import os
tdo=[x for x in os.listdir("outputs/phylogenetic_dataset") if "rev_aa.contree" in os.listdir(f"outputs/phylogenetic_dataset/{x}")]

In [ ]:
tdo.sort()

dfresult=pd.DataFrame()

In [ ]:
def get_data(i):
    try:
        t = Tree(f"outputs/phylogenetic_dataset/{i}/rev_aa_midpoint.treefile")

        with open(f"outputs/phylogenetic_dataset/{i}/phylo_dict.pkl", "rb") as f:
            l = pickle.load(f)

        tdn = list(l.keys())
        td = [
            x.split("; ")[-1].split(" x ")[0]
            if not x.split("; ")[-1].split(" x ")[0].isdigit()
            else x.split("; ")[-2].split(" x ")[0]
            for x in l.values()
        ]
        with mp.Pool(processes=50, initializer=init_worker) as pool:
            results = pool.map(get_ncbi_lineage_by_rank, td)
        lmap = dict(zip(tdn, [x[1] for x in results]))
        tmap= dict(zip(tdn, [x[0] for x in results]))

        for leaf in t:
            if leaf.name in lmap:
                l=lmap[leaf.name]
                leaf.add_features(sciarid= "Sciaridae" in str(l))
                leaf.add_features(metazoa= "Metazoa" in str(l) and 'Sciaridae' not in str(l))
                leaf.add_features(diptera= "Sciaridae" not in str(l) and 'Diptera' in str(l))
                leaf.add_features(taxid= tmap[leaf.name])

                if 'species' in l:
                    leaf.add_features(species=l['species'])

                if 'order' in l and l['order']=='Diptera':
                    if 'family' in l and l['family']=="Sciaridae":
                        leaf.add_features(tax_label= "Sciaridae")
                    else:

                        leaf.add_features(tax_label= "Diptera")
                elif 'class' in l and l['class']=='Insecta' :
                    leaf.add_features(tax_label=l['order'])
                elif 'phylum' in l and l['phylum']=='Arthropoda' :
                    leaf.add_features(tax_label=l['class'])
                elif 'kingdom' in l and l['kingdom']=='Metazoa':
                    leaf.add_features(tax_label=l['phylum'])
                elif 'Viruses' in str(l):
                    leaf.add_features(tax_label='Viruses')
                elif 'Fungi' in str(l):
                     leaf.add_features(tax_label='Fungi')
                elif 'Viridiplantae' in str(l):
                     leaf.add_features(tax_label='Viridiplantae')
                elif 'domain' in l and 'Eukaryota'==l['domain']:
                    leaf.add_features(tax_label='Protist')
                elif 'domain' in l and l['domain']=='Bacteria':
                    leaf.add_features(tax_label='Bacteria')
                elif 'domain' in l and l['domain']=='Archaea':
                    leaf.add_features(tax_label='Archaea')
                else:
                    leaf.add_features(tax_label='None')
        mono=[]
        for node in t.get_monophyletic(values=[True], target_attr="sciarid"):

            mono.append(node)
        accepted_nodes=[]
        rejected_nodes=[]
        for node in mono:
            accept = False
            node_og=node
            while node.up:

                check=len([x for x in t if x.tax_label=='Diptera'])>0
                if not check:
                    accepted_nodes.append(node_og)
                    break
                if node.up.support>70 and 'Diptera' not in set([x.tax_label for x in node.up]):
                    accepted_nodes.append(node_og)

                    break
                elif 'Diptera' in set([x.tax_label for x in node.up]):
                    rejected_nodes.append(node_og)
                    break
                node=node.up
        non_metazoan_transfer=False

        ni=1
        done=[]
        for node in accepted_nodes:
            if len(set([x.name for x in node])&htgs)>0:
                dfresult.loc[f"{i}.{ni}","all_seqs"]=str([x.name for x in node])
                dfresult.loc[f"{i}.{ni}","cluster_seqs"]=str(set([x.name for x in node])&set(connected_components[i]))
                dfresult.loc[f"{i}.{ni}","result"]='Accept'
                dfresult.loc[f"{i}.{ni}","n_cluster_seqs"]=len(set([x.name for x in node])&set(connected_components[i]))
                sciarid_species=list(set([x.species for x in node]))
                sciarid_species.sort()
                dfresult.loc[f"{i}.{ni}","sciarid_species"]=str(sciarid_species)

                sisters,cousins=get_sister_cousin(node)

                sister_taxlabel = {getattr(x, "tax_label", None) for x in sisters}
                cousin_taxlabel = {getattr(x, "tax_label", None) for x in cousins}

                if 'Sciaridae' in sister_taxlabel or 'Sciaridae' in cousin_taxlabel:
                    dfresult.loc[f"{i}.{ni}","sciarid_sister"]=True
                dfresult.loc[f"{i}.{ni}","sister_tax"]=str(sister_taxlabel-{'Sciaridae'})
                dfresult.loc[f"{i}.{ni}","cousin_tax"]=str(cousin_taxlabel-{'Sciaridae'})




                sister_tids=[x.taxid for x in sisters if x.tax_label!='Sciaridae']

                top=ncbi.get_topology(sister_tids)
                name=ncbi.get_taxid_translator([top.taxid])[top.taxid]
                dfresult.loc[f"{i}.{ni}","sister_name"]=str(name)
                l=ncbi.get_lineage(top.taxid)

                ranks={ncbi.get_rank([x])[x]:ncbi.get_taxid_translator([x])[x] for x in l}
                r=get_sister_info(ranks)
                dfresult.loc[f"{i}.{ni}",['sister_domain','sister_kingdom','sister_phylum','sister_class','sister_order']]=r

                non_metazoa={'Protist','Bacteria','Viridiplantae','Fungi', 'Viruses','Fungi','Archaea'}
                if len(cousin_taxlabel&non_metazoa)>0 or len(sister_taxlabel&non_metazoa)>0:
                    non_metazoan_transfer=True
                done.append(f"{i}.{ni}")

                paralogs=set([x.name for x in node if x.name in set(htgs)])
                if len(rejected_nodes)>0:
                    dfresult.loc[f"{i}.{ni}","pseudoparalog"]=True
                if len(paralogs)>1:
                     dfresult.loc[f"{i}.{ni}","post_transfer_duplication"]=True
                     dfresult.loc[f"{i}.{ni}","n_paralogs"]=len(paralogs)
                if len(accepted_nodes)>0:
                     dfresult.loc[f"{i}.{ni}","multi_transfer"]=True
                ni+=1
        for node in rejected_nodes:
            if len([ x for x in node if 'XP_' in x.name])>0:
                dfresult.loc[f"{i}.{ni}","all_seqs"]=str([x.name for x in node])
                dfresult.loc[f"{i}.{ni}","cluster_seqs"]=str(set([x.name for x in node])&set(connected_components[i]))
                dfresult.loc[f"{i}.{ni}","result"]='Reject'
                ni+=1
        if non_metazoan_transfer:
            mono=[]
            for node in t.get_monophyletic(values=[True], target_attr="metazoa"):
                mono.append(node)
            convergent_transfers=set()
            for y in mono:
                if len(set([x.tax_label for x in y]))==1:
                    sisters,cousins=get_sister_cousin(y)
                    sister_taxlabel = {getattr(x, "tax_label", None) for x in sisters}-{'Sciaridae'}
                    cousin_taxlabel = {getattr(x, "tax_label", None) for x in cousins}-{'Sciaridae'}
                    if len(sister_taxlabel&non_metazoa)>0 and len(cousin_taxlabel&non_metazoa)>0:
                        convergent_transfers=convergent_transfers|set([x.tax_label for x in y])
            dfresult.loc[done,'convergent']=str(convergent_transfers)
    except:
        print('fail',i)

In [ ]:
for x in tdo:

    get_data(int(x))

In [ ]:
len(connected_components)

924

In [ ]:
dfresult.to_csv('outputs/manual_inspection_table.tsv',sep='\t')

In [ ]:


def get_data(i):
    print(i)
    try:
        t = Tree(f"outputs/phylogenetic_dataset/{i}/rev_aa_midpoint.treefile")

        with open(f"outputs/phylogenetic_dataset/{i}/phylo_dict.pkl", "rb") as f:
            l = pickle.load(f)

        tdn = list(l.keys())
        td = [
            x.split("; ")[-1].split(" x ")[0]
            if not x.split("; ")[-1].split(" x ")[0].isdigit()
            else x.split("; ")[-2].split(" x ")[0]
            for x in l.values()
        ]
        with mp.Pool(processes=50, initializer=init_worker) as pool:
            results = pool.map(get_ncbi_lineage_by_rank, td)
        lmap = dict(zip(tdn, [x[1] for x in results]))
        tmap= dict(zip(tdn, [x[0] for x in results]))

        for leaf in t:
            if leaf.name in lmap:
                l=lmap[leaf.name]
                leaf.add_features(sciarid= "Sciaridae" in str(l))
                leaf.add_features(metazoa= "Metazoa" in str(l) and 'Sciaridae' not in str(l))
                leaf.add_features(diptera= "Sciaridae" not in str(l) and 'Diptera' in str(l))
                leaf.add_features(taxid= tmap[leaf.name])
                if 'species' in l:
                    leaf.add_features(species=l['species'])
                if 'order' in set(l.keys()) and l['order']=='Diptera':
                    if 'family' in l and l['family']=="Sciaridae":
                        leaf.add_features(tax_label= "Sciaridae")
                    else:

                        leaf.add_features(tax_label= "Diptera")
                elif 'class' in l and l['class']=='Insecta' :
                    leaf.add_features(tax_label=l['order'])
                elif 'phylum' in l and l['phylum']=='Arthropoda' :
                    leaf.add_features(tax_label=l['class'])
                elif 'kingdom' in l and l['kingdom']=='Metazoa':
                    leaf.add_features(tax_label=l['phylum'])
                elif 'Viruses' in str(l):
                    leaf.add_features(tax_label='Viruses')
                elif 'Fungi' in str(l):
                     leaf.add_features(tax_label='Fungi')
                elif 'Viridiplantae' in str(l):
                     leaf.add_features(tax_label='Viridiplantae')
                elif 'Eukaryota'==l['domain']:
                    leaf.add_features(tax_label='Protist')
                elif l['domain']=='Bacteria':
                    leaf.add_features(tax_label='Bacteria')
                elif l['domain']=='Archaea':
                    leaf.add_features(tax_label='Archaea')

        mono=[]
        for node in t.get_monophyletic(values=[True], target_attr="sciarid"):
            mono.append(node)
        accepted_nodes=[]
        rejected_nodes=[]
        for node in mono:
            accept = False
            # print(node_og)
            while node.up:

                check=len([x for x in t if x.tax_label=='Diptera'])>0
                if not check:
                    accepted_nodes.append(node)
                    break
                if node.up.support>70 and 'Diptera' not in set([x.tax_label for x in node.up]):
                    accepted_nodes.append(node)

                    break
                elif 'Diptera' in set([x.tax_label for x in node.up]):
                    rejected_nodes.append(node)
                    break
                node=node.up
        non_metazoan_transfer=False

        ni=1
        done=[]
        for node in accepted_nodes:
            if len([ x for x in node if 'XP_' in x.name])>0:
                dfresult.loc[f"{i}.{ni}","all_seqs"]=str([x.name for x in node])
                dfresult.loc[f"{i}.{ni}","cluster_seqs"]=str(set([x.name for x in node])&set(connected_components[i]))
                dfresult.loc[f"{i}.{ni}","result"]='Accept'
                dfresult.loc[f"{i}.{ni}","n_cluster_seqs"]=len(set([x.name for x in node])&set(connected_components[i]))
                dfresult.loc[f"{i}.{ni}","sciarid_species"]=len(set([x.species for x in node]))

                sisters,cousins=get_sister_cousin(node)
                sister_taxlabel = {getattr(x, "tax_label", None) for x in sisters}-{'Sciaridae'}
                cousin_taxlabel = {getattr(x, "tax_label", None) for x in cousins}-{'Sciaridae'}
                dfresult.loc[f"{i}.{ni}","sister_tax"]=str(sister_taxlabel)
                dfresult.loc[f"{i}.{ni}","cousin_tax"]=str(cousin_taxlabel)




                sister_tids=[x.taxid for x in sisters if x.tax_label!='Sciaridae']
                top=ncbi.get_topology(sister_tids)
                name=ncbi.get_taxid_translator([top.taxid])[top.taxid]
                dfresult.loc[f"{i}.{ni}","sister_name"]=str(name)
                l=ncbi.get_lineage(top.taxid)

                ranks={ncbi.get_rank([x])[x]:ncbi.get_taxid_translator([x])[x] for x in l}
                r=get_sister_info(ranks)
                dfresult.loc[f"{i}.{ni}",['sister_domain','sister_kingdom','sister_phylum','sister_class','sister_order']]=r

                non_metazoa={'Protist','Bacteria','Viridiplantae','Fungi', 'Viruses','Fungi','Archaea'}
                if len(cousin_taxlabel&non_metazoa)>0 or len(sister_taxlabel&non_metazoa)>0:
                    non_metazoan_transfer=True
                done.append(f"{i}.{ni}")

                paralogs=set([x.name for x in node if x.name in set(htgs)])
                if len(rejected_nodes)>0:
                    dfresult.loc[f"{i}.{ni}","pseudoparalog"]=True
                if len(paralogs)>0:
                     dfresult.loc[f"{i}.{ni}","post_transfer_duplication"]=True
                     dfresult.loc[f"{i}.{ni}","n_paralogs"]=len(paralogs)
                if len(accepted_nodes)>0:
                     dfresult.loc[f"{i}.{ni}","multi_transfer"]=True
                ni+=1
        for node in rejected_nodes:
            if len([ x for x in node if 'XP_' in x.name])>0:
                dfresult.loc[f"{i}.{ni}","all_seqs"]=str([x.name for x in node])
                dfresult.loc[f"{i}.{ni}","cluster_seqs"]=str(set([x.name for x in node])&set(connected_components[i]))
                dfresult.loc[f"{i}.{ni}","result"]='Reject'
                ni+=1
        if non_metazoan_transfer:
            mono=[]
            for node in t.get_monophyletic(values=[True], target_attr="metazoa"):
                mono.append(node)
            convergent_transfers=set()
            for y in mono:
                if len(set([x.tax_label for x in y]))==1:
                    sisters,cousins=get_sister_cousin(y)
                    sister_taxlabel = {getattr(x, "tax_label", None) for x in sisters}-{'Sciaridae'}
                    cousin_taxlabel = {getattr(x, "tax_label", None) for x in cousins}-{'Sciaridae'}
                    if len(sister_taxlabel&non_metazoa)>0 and len(cousin_taxlabel&non_metazoa)>0:
                        convergent_transfers=convergent_transfers|set([x.tax_label for x in y])
            dfresult.loc[done,'convergent']=str(convergent_transfers)
    except:
        print("fail",i)



0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
20
21
22
23
24
25
26
fail 26
27
fail 27
28
29
30
31
32
33
34
35
36
37
38
39
40
40
fail 40
41
42
42
43
44
45
46
47
48
49
50
51
52
53
53
54
55
56
fail 56
57
58
59
60
61
62
63
64
64
65
66
67
68
69
70
71
72
73
74
75
76
77
77
78
79
80
81
81
82
83
84
85
86
87
88
89
90
91
92
93
fail 93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
111
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
131
132
133
134
134
135
136
fail 136
137
138
139
140
141
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
157
158
159
160
161
162
163
164
164
165
166
167
168
169
170
171
172
173
173
174
175
176
177
178
179
fail 179
180
181
fail 181
182
183
184
185
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
fail 239
240
241
242
243
244
245
246
2

## Phylostratigraphy

In [ ]:
def get_sciar_species(x):
    if 'XP_' in x:
        return 'Bradysia coprophila'
    elif 'lingenua' in x:
        return 'Lycoriella ingenua'
    elif 'bimpatiens' in x:
        return 'Bradysia impatiens'
    elif 'KAJ' in x:
        return 'Pseudolycoriella hygida'
    elif 'KAG' in x:
        return 'Bradysia odoriphaga'

In [4]:
import pandas as pd
df=pd.read_csv("outputs/SI_inspection_bradysia_6_23_2026.tsv",sep="\t",index_col=0)

In [6]:
for index, row in df.iterrows():
    tree=str(index).split(".")[0]
    all_seqs=set([x.strip() for x in row.all_seqs.split(",")])
    cluster_seqs=connected_components[int(tree)]&all_seqs
    df.loc[index,['cluster_seqs','n_cluster_seqs']]=cluster_seqs,len(cluster_seqs)
    sciarid_species=list(set([get_sciar_species(x) for x in all_seqs]))
    sciarid_species.sort()
    df.loc[index,'sciarid_species']=str(sciarid_species).replace("'","").replace("[","").replace("]","")
    if 'Pseudolycoriella hygida' in sciarid_species:
        age=5
    elif 'Lycoriella ingenua' in sciarid_species:
        age=4
    elif 'Bradysia impatiens' in sciarid_species:
        age=3
    elif 'Bradysia odoriphaga' in sciarid_species:
        age=2
    else:
        age=1
    df.loc[index, 'age']=age


In [ ]:
# keep original index as string
old_index = df.index.astype(str)

# get the number before "."
major = old_index.str.split(".").str[0]

# assign 1, 2, 3... within each major group
minor = df.groupby(major).cumcount() + 1

# create new standardized index
df.index = major + "." + minor.astype(str)



In [ ]:
import ast
tree_htgs=[]
for index, row in df[~df.annotation.isin(['Reject','Other direction'])].iterrows():
    tree_htgs.extend([x.strip() for x in ast.literal_eval(row.cluster_seqs)])

In [ ]:
## print age category, n_transfers, n_htgs
for age, dfi in  df[~df.annotation.isin(['Reject','Other direction'])].groupby('age'):
    print(age,dfi.shape[0],dfi.n_cluster_seqs.sum())

1.0 45 53
2.0 27 28
3.0 79 106
4.0 250 330
5.0 533 1000


In [ ]:
## total number of htgs
tree_htgs=set(tree_htgs)
len(tree_htgs)

1517

In [ ]:
dfprot=pd.read_csv("outputs/per_protein_diamond_data.tsv",sep="\t",index_col=0)
dfprot['tree_HTG']=[x in tree_htgs for x in dfprot.accession]
dfprot.to_csv("outputs/per_protein_diamond_data.tsv",sep="\t")

In [8]:
## total number of transfers
htgdf=df[~df.annotation.isin(['Reject','Other direction'])]
htgdf.shape[0]

934

In [ ]:
from ete3 import Tree
for index, row in df[~df.annotation.isin(['Reject','Other direction'])].iterrows():
    i=str(index).split(".")[0]
    t = Tree(f"outputs/phylogenetic_dataset/{i}/rev_aa_midpoint.treefile")
    seqs=set([x.name for x in t])


    ##multitransfer candidates have htgs in other clades
    htgss=seqs&tree_htgs-set([x.strip() for x in row.all_seqs.split(",")])



    df.loc[index,'multi_transfer']=len(htgss)>0

    # to be corrected by manual inspection
    if row.annotation=='One step' and row['primary neighbor'] not in  ['Bacteria', 'Fungi', 'Protist','Viridiplantae','Arachaea','Viruses']:
        dipts=seqs&dipt_proteins
        df.loc[index,'pseudoparalog']=len(dipts)>0


In [ ]:
df.columns=['all_seqs', 'n_cluster_seqs', 'cluster_seqs', 'sciarid_species',
       'annotation', 'primary neighbor', 'secondary neighbor', 're-root',
       'notes', 'convergent', 'pseudoparalog', 'multi_transfer', 'age']

In [ ]:
## number of manually re-rooted
len(set([str(x).split(".")[0] for x in df[df['re-root'].astype(str)!='nan'].index]))

55

## Post-transfer duplication

In [9]:
## number/percentage of sequences in multi-gene families
htgdf[htgdf.n_cluster_seqs>1].n_cluster_seqs.sum(),htgdf[htgdf.n_cluster_seqs>1].n_cluster_seqs.sum()/htgdf.n_cluster_seqs.sum()


(np.int64(830), np.float64(0.5471324983520105))

In [10]:
## number/percentage of hgt events in multi-gene families
htgdf[htgdf.n_cluster_seqs>1].shape[0],htgdf[htgdf.n_cluster_seqs>1].shape[0]/htgdf.shape[0]

(247, 0.2644539614561028)

In [11]:
from collections import Counter
pdf=pd.DataFrame.from_dict(Counter(htgdf.n_cluster_seqs),orient='index')
pdf.index.name='Number of paralogs'
pdf.columns=['Number of transfers']
pdf.sort_index()

,Number of transfers
Number of paralogs,
1,687
2,127
3,54
4,24
5,16
6,7
7,6
8,4
9,3


## Multiple independent transfers

In [12]:
## number/percentage of sequences in multi-gene families
htgdf[htgdf.multi_transfer==True].n_cluster_seqs.sum(),htgdf[htgdf.multi_transfer==True].n_cluster_seqs.sum()/htgdf.n_cluster_seqs.sum()


(np.int64(631), np.float64(0.4159525379037574))

In [13]:
## number/percentage of hgt events in multi-gene families
htgdf[htgdf.multi_transfer==True].shape[0],htgdf[htgdf.multi_transfer==True].shape[0]/htgdf.shape[0]


(364, 0.3897216274089936)

## Pseudoparalogs


In [14]:
## number/percentage of sequences with pseudoparalogs
htgdf[htgdf.pseudoparalog==True].n_cluster_seqs.sum(),htgdf[htgdf.pseudoparalog==True].n_cluster_seqs.sum()/htgdf.n_cluster_seqs.sum()


(np.int64(388), np.float64(0.25576796308503624))

In [15]:
## number/percentage of hgt events with pseudoparalogs
htgdf[htgdf.pseudoparalog==True].shape[0],htgdf[htgdf.pseudoparalog==True].shape[0]/htgdf.shape[0]


(255, 0.27301927194860814)

In [ ]:
from collections import Counter
Counter(df[~df.annotation.isin(['Reject','Other direction'])]['age'])

Counter({5.0: 533, 4.0: 250, 1.0: 45, 2.0: 27, 3.0: 79})

In [18]:
annot_df=pd.DataFrame(index=['Two step unclear direction',
 'Reject',
 'Two step',
 'One step metazoan',
'One step non-metazoan',
 'Other direction',
 'Ambiguous',
 'Two step other direction'],columns=['N_HTGs','N_clades']).fillna(0)
for index, row in df.iterrows():
    if row.annotation=='One step':
        if row['primary neighbor'] in ['Archaea','Bacteria','Fungi','Protist','Viridiplantae','Viruses']:
            a='One step non-metazoan'
        else:
            a='One step metazoan'
    else:
        a=row.annotation
    annot_df.loc[a,'N_HTGs']+=row.n_cluster_seqs
    annot_df.loc[a,'N_clades']+=1
annot_df['p_HTGs']=100*round(annot_df['N_HTGs']/annot_df[~annot_df.index.isin(['Reject','Other direction'])]['N_HTGs'].sum(),3)
annot_df['p_clades']=100*round(annot_df['N_clades']/annot_df[~annot_df.index.isin(['Reject','Other direction'])]['N_clades'].sum(),3)

/tmp/ipykernel_1478/1050996575.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  'Two step other direction'],columns=['N_HTGs','N_clades']).fillna(0)


In [20]:
(448+117)/934

0.6049250535331906

In [19]:
annot_df

,N_HTGs,N_clades,p_HTGs,p_clades
Two step unclear direction,93,65,6.1,7.0
Reject,177,87,11.7,9.3
Two step,194,102,12.8,10.9
One step metazoan,710,448,46.8,48.0
One step non-metazoan,171,117,11.3,12.5
Other direction,21,8,1.4,0.9
Ambiguous,304,192,20.0,20.6
Two step other direction,45,10,3.0,1.1


In [ ]:
pdf=pd.DataFrame.from_dict(Counter(df[df['annotation']=='One step']['primary neighbor']),orient='index')
pdf.index.name='Taxon of origin'
pdf.columns=['Number of transfers']
pdf.sort_values('Number of transfers',ascending=False)

,Number of transfers
Taxon of origin,
Collembola,304
Arachnida,80
Fungi,60
Bacteria,41
Rotifera,16
Protist,14
Hymenoptera,13
Coleoptera,11
Hemiptera,8


In [ ]:
pdf=pd.DataFrame.from_dict(Counter(df[df['annotation']=='Two step']['primary neighbor']),orient='index')
pdf.index.name='Taxon of origin'
pdf.columns=['Number of transfers']
pdf.sort_values('Number of transfers',ascending=False)

,Number of transfers
Taxon of origin,
Collembola,82
Rotifera,14
Arachnida,6


In [ ]:
pdf=pd.DataFrame.from_dict(Counter(df[df['annotation']=='Two step']['secondary neighbor']),orient='index')
pdf.index.name='Taxon of origin'
pdf.columns=['Number of transfers']
pdf.sort_values('Number of transfers',ascending=False)

,Number of transfers
Taxon of origin,
Fungi,44
Bacteria,39
Ambiguous,14
Protist,4
Viruses,1


In [ ]:
pdf=pd.DataFrame.from_dict(Counter(df[df['annotation']=='Two step unclear direction']['primary neighbor']),orient='index')
pdf.index.name='Taxon of origin'
pdf.columns=['Number of transfers']
pdf.sort_values('Number of transfers',ascending=False)

,Number of transfers
Taxon of origin,
Collembola,50
Rotifera,12
Arachnida,3


In [ ]:
pdf=pd.DataFrame.from_dict(Counter(df[df['annotation']=='Two step unclear direction']['secondary neighbor']),orient='index')
pdf.index.name='Taxon of origin'
pdf.columns=['Number of transfers']
pdf.sort_values('Number of transfers',ascending=False)

,Number of transfers
Taxon of origin,
Bacteria,24
Fungi,21
Protist,12
Ambiguous,7
Viridiplantae,1


In [ ]:
pdf=pd.DataFrame.from_dict(Counter(df[df['annotation']=='Two step other direction']['primary neighbor']),orient='index')
pdf.index.name='Taxon of origin'
pdf.columns=['Number of transfers']
pdf.sort_values('Number of transfers',ascending=False)

,Number of transfers
Taxon of origin,
Collembola,8
Rotifera,2


In [ ]:
pdf=pd.DataFrame.from_dict(Counter(df[df['annotation']=='Two step other direction']['secondary neighbor']),orient='index')
pdf.index.name='Taxon of origin'
pdf.columns=['Number of transfers']
pdf.sort_values('Number of transfers',ascending=False)

,Number of transfers
Taxon of origin,
Bacteria,6
Fungi,2
Protist,2


In [ ]:
pdf=pd.DataFrame.from_dict(Counter(df[df['annotation']=='Other direction']['primary neighbor']),orient='index')
pdf.index.name='Taxon of origin'
pdf.columns=['Number of transfers']
pdf.sort_values('Number of transfers',ascending=False)

,Number of transfers
Taxon of origin,
Collembola,4
Bacteria,1
Branchiopoda,1
Arachnida,1
Archaea,1


## Convergent transfers to other metazoans

In [ ]:
htgdf=df[~df.annotation.isin(['Reject','Other direction'])]
c=htgdf[htgdf.convergent.astype(str)!='nan']
from collections import defaultdict

condict=defaultdict(int)
for index, row in c.iterrows():
    for x in row.convergent.split(","):
        condict[x.strip()]+=1
condf=pd.DataFrame.from_dict(condict,orient='index').sort_values(0,ascending=False)
condf.index.name='Recipient taxon'
condf.columns=['Number of transfers']
condf.to_csv("outputs/SI_tables/SI_table_9_convergent_transfers.tsv",sep="\t")

In [ ]:

condf['Number of transfers'].sum()

194

In [ ]:
## number of transfer events with multiple independent transfers
c[c.convergent.str.contains(",")].shape[0]

46

In [ ]:
## number and percentage of transfers with a convergent transfer
htgdf[htgdf.convergent.astype(str)!='nan'].shape[0],htgdf[htgdf.convergent.astype(str)!='nan'].shape[0]/htgdf.shape[0]

(128, 0.13704496788008566)

In [ ]:
## number and percentage of transfers with a convergent transfer
htgdf[htgdf.convergent.astype(str)!='nan'].n_cluster_seqs.sum(),htgdf[htgdf.convergent.astype(str)!='nan'].n_cluster_seqs.sum()/htgdf.n_cluster_seqs.sum()

(248, 0.16348055372445616)

In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv("outputs/SI_inspection_bradysia_6_23_2026.tsv",sep="\t",index_col=0)

In [ ]:
htgdf=df[~df.annotation.isin(['Reject','Other direction'])]


In [ ]:
set(htgdf[htgdf.convergent.astype(str)!='nan']['annotation'])

{'One step',
 'Two step',
 'Two step other direction',
 'Two step unclear direction'}